In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Load the Datasets
# ==========================================
customers = pd.read_csv('customers_data.csv')
transactions = pd.read_csv('transactions_data.csv')
products = pd.read_csv('products_data.csv')

# ==========================================
# 2. Preprocessing & Cleaning
# ==========================================

def clean_products(df):
    """Clean product prices and impute IDs."""
    # Convert '?140,000' strings to numeric float
    df['Product_Price'] = df['Product_Price'].astype(str).str.replace('?', '', regex=False).str.replace(',', '', regex=False).astype(float)
    # Fill missing Product IDs with sequential numbers
    df['Product_ID'] = df['Product_ID'].fillna(pd.Series(range(1, len(df) + 1), index=df.index)).astype(int)
    return df

def clean_customers(df):
    """Clean company names, impute profit, and extract location features."""
    # Standardize names (remove double spaces)
    df['Company_Name'] = df['Company_Name'].str.strip().replace(r'\s+', ' ', regex=True)
    
    # Impute Company_ID by mapping names to existing IDs
    id_map = df.dropna(subset=['Company_ID']).set_index('Company_Name')['Company_ID'].to_dict()
    df['Company_ID'] = df['Company_ID'].fillna(df['Company_Name'].map(id_map))
    df['Company_ID'] = df['Company_ID'].fillna(pd.Series(range(1, len(df) + 1), index=df.index)).astype(int)
    
    # Impute Profit with median
    df['Company_Profit'] = df['Company_Profit'].fillna(df['Company_Profit'].median())
    
    # Extract City from Address (e.g., "Pasig" from "...Pasig, Philippines")
    df['City'] = df['Address'].apply(lambda x: x.split(',')[-2].strip() if len(x.split(',')) >= 2 else 'Unknown')
    return df

def clean_transactions(df, products_df):
    """Standardize dates and recover missing cost/quantity data."""
    # Drop irrelevant system columns
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    
    # Standardize inconsistent date formats
    df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format='mixed', dayfirst=False, errors='coerce')
    
    # Recover missing Product_Price using the products master list
    price_map = products_df.set_index('Product_ID')['Product_Price'].to_dict()
    df['Product_Price'] = df['Product_Price'].fillna(df['Product_ID'].map(price_map))
    
    # Impute missing Quantity (assume 1) and calculate Total_Cost
    df['Quantity'] = df['Quantity'].fillna(1.0)
    df['Total_Cost'] = df['Total_Cost'].fillna(df['Quantity'] * df['Product_Price'])
    
    # Remove records where critical IDs (Company/Product) are completely missing
    df = df.dropna(subset=['Company_ID', 'Product_ID'])
    df[['Company_ID', 'Product_ID']] = df[['Company_ID', 'Product_ID']].astype(int)
    return df

# Apply cleaning
products_cleaned = clean_products(products.copy())
customers_cleaned = clean_customers(customers.copy())
transactions_cleaned = clean_transactions(transactions.copy(), products_cleaned)

# ==========================================
# 3. Feature Engineering (RFM Analysis)
# ==========================================

# Reference date for Recency (Day after the last transaction in dataset)
ref_date = transactions_cleaned['Transaction_Date'].max() + pd.Timedelta(days=1)

# Aggregate transaction data per customer
customer_metrics = transactions_cleaned.groupby('Company_ID').agg(
    Recency=('Transaction_Date', lambda x: (ref_date - x.max()).days),
    Frequency=('Transaction_ID', 'count'),
    Monetary=('Total_Cost', 'sum'),
    Avg_Quantity=('Quantity', 'mean')
).reset_index()

# Merge metrics back into the customer profile
final_df = pd.merge(customers_cleaned, customer_metrics, on='Company_ID', how='left')

# Handle customers with zero transactions
final_df['Frequency'] = final_df['Frequency'].fillna(0)
final_df['Monetary'] = final_df['Monetary'].fillna(0)
final_df['Recency'] = final_df['Recency'].fillna(final_df['Recency'].max())

# Add predictive features
# 1. High Value Flag: Top 25% of spenders
final_df['Is_High_Value'] = (final_df['Monetary'] > final_df['Monetary'].quantile(0.75)).astype(int)

# 2. Profit Efficiency Ratio
final_df['Profit_per_Monetary'] = final_df['Company_Profit'] / (final_df['Monetary'] + 1)

# Export the polished dataset
final_df.to_csv('final_customer_purchase_behavior.csv', index=False)